- select a robust universe of ortagonal return sourcet e.g. (Bonds, Commodity ETFs, by sector ETFs, etc...
- filter by orthagonality/correlation
- from remaining assets construct a portfolio using Entropy pooling
- test on historical returns(get ES data from the most recent market crash)



In [ ]:
pip install ecos

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 10.0 MB/s eta 0:00:00


In [176]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import os
from dataclasses import dataclass, field
from typing import Optional, Sequence, Union
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
import cvxpy as cp

In [ ]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def download(self, indices:str, start:str, end:str, interval:str):
    data = {}
    for ticker in indices:
        clean_ticker = ticker.strip()
        try:
            tmp_data = yf.Ticker(clean_ticker) \
                          .history(
                              start=start,
                              end=end,
                              interval=interval
                          )["Close"]

            if tmp_data.empty:
                print(f"Warning: No data returned for {clean_ticker}")
                continue
            tmp_data.index = pd.to_datetime(tmp_data.index)\
                                    .tz_localize(None)\
                                    .normalize()
            tmp_data = tmp_data[~tmp_data.index.duplicated(keep='last')]
            data[clean_ticker] = tmp_data
        except Exception as e:
            print(f"Failed to download {clean_ticker}: {e}")

    if not data:
        raise ValueError(
            "No data was successfully downloaded  \
              for any of the provided tickers."
        )
    df = pd.concat(data, axis=1)
    return df.sort_index()

  def get_data(
      self,
      tickers:list,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"
    benchmark_path = f"benchmark_{start}_{end}.parquet"

    if not os.path.exists(df_path) or not os.path.exists(benchmark_path):
      df = yf.download(tickers, start, end, interval)["Close"]
      bench_data = yf.download([benchmark], start, end, interval)["Close"]

      df.to_parquet(df_path)
      bench_data.to_parquet(benchmark_path)

    else:
      df = pd.read_parquet(df_path)
      bench_data= pd.read_parquet(benchmark_path)

    returns_raw = df.pct_change().dropna()
    benchmark = bench_data.pct_change().dropna()
    self.universe = returns_raw.columns

    return returns_raw, benchmark

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [ ]:
class Optimizer:
  def __init__(
      self,
      optim_params,
      debug:bool=False,
      **kwargs
  ):
    self.debug = debug
    self.es_prct = optim_params.es_prct
    self.turnover_penalty = optim_params.turnover_penalty
    self.risk_pen = optim_params.risk_penalty
    self.tail_penalty = optim_params.tail_penalty

  def optimize_lambda(self, p, A, b, k_eq, k_ineq):
    constraints = []
    K = k_eq + k_ineq
    lambda_var = cp.Variable(K)

    if k_ineq > 0:
        constraints.append(lambda_var[k_eq:] >= 0)

    exp_terms = -lambda_var @ A + np.log(p)
    obj_fn = cp.log_sum_exp(exp_terms) + lambda_var @ b

    prob = cp.Problem(cp.Minimize(obj_fn), constraints)
    prob.solve(solver=cp.ECOS)

    if prob.status not in ["optimal", "optimal_inaccurate"]:
        raise RuntimeError(f"Optimization failed. Status: {prob.status}")

    return lambda_var.value

  def optimize_w(self, returns, mean, covariance, w_prev=None):
    R = returns.values
    S, N = R.shape
    w = cp.Variable(N)
    u = cp.Variable(S)
    zeta = cp.Variable()

    mu = mean.values
    cov = covariance.values

    es = zeta + (1 / ((1-self.es_prct) * S)) * cp.sum(u)

    constraints = [
        u >= -R @ w - zeta,
        u >= 0,
        cp.sum(w) == 1,
        w >= 0
    ]

    ex_r = mu @ w

    if w_prev is not None:
      turnover_penalty = self.turnover_penalty * cp.sum((w - w_prev)**2)

    else:
      turnover_penalty = 0

    risk_term = cp.quad_form(w, cov)
    obj_fn = cp.Maximize(ex_r - risk_term - es - turnover_penalty)

    prob = cp.Problem(obj_fn, constraints)
    prob.solve(solver=cp.CLARABEL)

    if prob.status not in ["optimal", "optimal_inaccurate"]:
        raise ValueError(f"GMV optimization failed: {prob.status}")

    return w.value



In [ ]:
class Portfolio(Optimizer, DataStore):
  def __init__(self, optim_params, debug:bool=False, **kwargs):
    self.debug = debug
    super().__init__(debug=debug, optim_params=optim_params, **kwargs)

  def _add(self, A, b, vtype, direction="le"):
    if vtype == "ineq":
      if direction == "ge":
        self.A_ineq.append(-A)
        self.b_ineq.append(-b)
      else:
        self.A_ineq.append(A)
        self.b_ineq.append(b)
    elif vtype == "eq":
      self.A_eq.append(A)
      self.b_eq.append(b)

  def _compute_window_returns(self, returns, window):
    if window <= 1:
      return returns

    log_r = np.log1p(returns)
    roll_sum = log_r.rolling(window).sum()
    window_returns = np.expm1(roll_sum).dropna()

    return window_returns

  def get_views(self, views, returns, p):
    self.A_eq, self.b_eq, self.A_ineq, self.b_ineq = [], [], [], []

    R = returns.values
    for view in views:
      if view["view"] == "mean":
        idx = returns.columns.get_loc(view["ticker"])

        self._add(R[:, idx], view["value"], view["type"], view.get("direction", "le"))

      elif view["view"] == "volatility":
        idx = returns.columns.get_loc(view["ticker"])
        R_ = R[:, idx]
        mu = np.sum(p*R_)
        var = (R_ - mu)**2

        self._add(var, view["value"]**2, view["type"], view.get("direction", "le"))

      elif view["view"] == "relative":
        idx1 = returns.columns.get_loc(view["ticker1"])
        idx2 = returns.columns.get_loc(view["ticker2"])

        A_rel = R[:, idx1] - R[:, idx2]
        self._add(A_rel, view["value"], view["type"], view.get("direction", "le"))

      elif view["view"] == "correlation":
        idx1 = returns.columns.get_loc(view["ticker1"])
        idx2 = returns.columns.get_loc(view["ticker2"])

        mu_1 = np.sum(p*R[:, idx1])
        mu_2 = np.sum(p*R[:, idx2])

        vol_1 = np.sqrt(np.sum(p*(R[:, idx1] - mu_1)**2))
        vol_2 = np.sqrt(np.sum(p*(R[:, idx2] - mu_2)**2))
        rho = view["value"]
        cov_target = rho*vol_1*vol_2

        A_corr = (R[:, idx1] - mu_1)*(R[:, idx2] - mu_2)
        self._add(A_corr, cov_target, view["type"], view.get("direction", "le"))

  def apply_lot_sizing(self, weights, asset_prices, total_portfolio_value=1000):
    prices = np.array(asset_prices, dtype=float)

    valid = (prices > 0) & np.isfinite(prices)
    if not valid.all():
      weights = weights.copy()
      weights[~valid] = 0.0
      total = weights.sum()
      if total > 0:
        weights = weights / total
      prices = np.where(valid, prices, 1.0)

    ideal_cash_alloc = weights * total_portfolio_value
    ideal_shares = ideal_cash_alloc / prices
    actual_shares = np.round(ideal_shares)
    actual_shares[~valid] = 0.0
    actual_cash_alloc = actual_shares * prices
    actual_total_spent = np.sum(actual_cash_alloc)

    if actual_total_spent > 0:
        rounded_weights = actual_cash_alloc / actual_total_spent
    else:

        rounded_weights = weights

    return rounded_weights, actual_shares

  def solve_entropy_pooling(
    self,
    R,
    views=None,
    p=None,
    window=1
  ):
    R_calc = self._compute_window_returns(R, window)
    S, N = R_calc.shape

    if p is None:
      p = np.ones(S) / S
    else:
      p = np.asarray(p)
      p = p[-S:]
      p = p / p.sum()

    if views is not None:
        self.get_views(views, R_calc, p)
    else:
      self.A_eq, self.b_eq, self.A_ineq, self.b_ineq = None, None, None, None

    A_list, b_list = [], []
    k_eq = 0

    if self.A_eq is not None and self.b_eq is not None and len(self.A_eq) > 0:
      A_eq_clean = np.atleast_2d(self.A_eq)
      b_eq_clean = np.atleast_1d(self.b_eq)
      A_list.append(A_eq_clean)
      b_list.append(b_eq_clean)
      k_eq = A_eq_clean.shape[0]

    k_ineq = 0
    if self.A_ineq is not None and self.b_ineq is not None and len(self.A_ineq) > 0:
      A_ineq_clean = np.atleast_2d(self.A_ineq)
      b_ineq_clean = np.atleast_1d(self.b_ineq)
      A_list.append(A_ineq_clean)
      b_list.append(b_ineq_clean)
      k_ineq = A_ineq_clean.shape[0]

    A = np.vstack(A_list)
    b = np.concatenate(b_list)

    opt_lambda = self.optimize_lambda(p, A, b, k_eq, k_ineq)

    q = p * np.exp(-opt_lambda @ A)
    q /= np.sum(q)
    mu_post = q @ R_calc.values
    R_dev = R_calc.values - mu_post
    cov_post = (R_dev.T * q) @ R_dev

    mu_post = pd.Series(mu_post, index=R_calc.columns)
    cov_post = pd.DataFrame(cov_post, index=R_calc.columns, columns=R_calc.columns)

    return mu_post, cov_post


  def optimize_portfolio(self, returns, views, p=None, window=1, w_prev=None):
    S, N = returns.shape

    p = p if p is not None else np.ones(S)/S
    print("---------------------- Entropy Pooling ----------------------")
    mu_post, cov_post = self.solve_entropy_pooling(returns, views, p, window=window)
    print(f"posterior mean: \n {mu_post}")
    print(f"posterior covariance: \n {cov_post}")

    print("------------------ Optimizing Portfolio w -------------------")
    w_raw = self.optimize_w(returns, mu_post, cov_post, w_prev=w_prev)

    w_opt = self.apply_lot_sizing(w_raw, returns.iloc[-1]) if self.round_weights else w_raw

    return w_opt

In [ ]:
class BatesModel:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )

  def generate_path(self, N_paths, S):
    pass

In [ ]:
class Backtest(Portfolio):
  def __init__(self, optim_params, debug:bool=False, **kwargs):
    self.debug = debug
    super().__init__(debug=debug, optim_params=optim_params, **kwargs)

  def generate_path(self, N_paths, S):
    W = np.random.normal

In [179]:
from dataclasses import dataclass, asdict
@dataclass
class OptimParams:
  es_prct: float
  turnover_penalty: float
  risk_penalty: float
  tail_penalty: float


In [178]:
test_views = [
    {
        "view": "mean",
        "ticker": "QLD",
        "value": 0.08,
        "type": "eq"
    },
    {
        "view": "relative",
        "ticker1": "KMLM",
        "ticker2": "BNDW",
        "value": 0.04,
        "type": "ineq",
        "direction": "ge"
    }
]

In [180]:
tickers = [
  "QLD",
  "BNDW",
  "KMLM"
]

optim_params = OptimParams(
    es_prct=0.95,
    turnover_penalty=1,
    risk_penalty=2,
    tail_penalty=1.5
)

In [ ]:
ptf = Portfolio(optim_params, debug=False)

In [ ]:
returns, benchmark = ptf.get_data(
    tickers=tickers,
    start="2020-01-01",
    end="2026-01-01",
    interval="1d"
)

In [ ]:
mu_post, cov_post = ptf.solve_entropy_pooling(R=returns, views=test_views, window=21)

In [ ]:
ArrayLike = Union[float, Sequence[float], np.ndarray]

@dataclass
class BatesParams:
    N: int = 1000
    S: int = 252
    N_assets: int = 1 
    T: int = 252

    rfr: ArrayLike = 0.003
    div_yield: ArrayLike = 0.0
    corr: ArrayLike = -0.7 
    jump_intensity: ArrayLike = 0.5
    mu_j: ArrayLike = -0.05
    sigma_j: ArrayLike = 0.10

    asset_corr: Optional[np.ndarray] = None

    @property
    def dt(self) -> float:
      return 1 / self.T

    def __post_init__(self):
      n = self.N_assets
      self.rfr = self._as_asset_array(self.rfr, n)
      self.div_yield = self._as_asset_array(self.div_yield, n)
      self.corr = self._as_asset_array(self.corr, n)
      self.jump_intensity = self._as_asset_array(self.jump_intensity, n)
      self.mu_j = self._as_asset_array(self.mu_j, n)
      self.sigma_j = self._as_asset_array(self.sigma_j, n)

      if self.asset_corr is not None:
        self.asset_corr = np.asarray(self.asset_corr, dtype=float)
        if self.asset_corr.shape != (n, n):
          raise ValueError(
            f"asset_corr must have shape ({n}, {n}), got {self.asset_corr.shape}"
          )

    @staticmethod
    def _as_asset_array(value: ArrayLike, n: int) -> np.ndarray:
      arr = np.asarray(value, dtype=float)
      if arr.ndim == 0:
        return np.full(n, arr)
      if arr.shape != (n,):
        raise ValueError(f"expected scalar or shape ({n},) array, got shape {arr.shape}")
      return arr

In [ ]:
class BatesModel:
    def __init__(self, params: BatesParams):
      self.params = params
      self.V = None
      self.Z_V = None
      self.Z_S = None
      self.log_jumps = None
      self.S_paths = None
      self._asset_chol = None 

    @staticmethod
    def garman_klass_vol(asset_data: pd.DataFrame, lambda_param: float = 0.94) -> np.ndarray:
      ln_CO = np.log(asset_data["Close"] / asset_data["Open"])
      ln_HL = np.log(asset_data["High"] / asset_data["Low"])
      gk_var = 0.5 * (ln_HL ** 2) - (2 * np.log(2) - 1) * (ln_CO ** 2)

      T = len(gk_var)
      smoothed_variance = np.zeros(T)
      smoothed_variance[0] = gk_var.iloc[0]
      for t in range(1, T):
        smoothed_variance[t] = (
          lambda_param * smoothed_variance[t - 1]
          + (1 - lambda_param) * gk_var.iloc[t]
        )
      return np.sqrt(smoothed_variance * 252)

    @staticmethod
    def get_mr_params(vix, dt: float):
        if isinstance(vix, pd.DataFrame):
          vix = vix["Close"] if "Close" in vix.columns else vix.iloc[:, 0]
        vix = vix.squeeze() if hasattr(vix, "squeeze") else vix

        vix_var_data = (vix / 100) ** 2
        model = AutoReg(vix_var_data, lags=1, trend="c")
        model_fit = model.fit()

        c = model_fit.params["const"]
        phi = model_fit.params.iloc[1]

        phi_1 = 1 - phi
        mr_speed = phi_1 / dt
        lt_var = c / phi_1

        resid = model_fit.resid
        full_lag_var = vix_var_data.shift(1)
        lag_var = full_lag_var.loc[model_fit.resid.index].values

        scaled_resid = resid.values / np.sqrt(lag_var)

        var = np.var(scaled_resid, ddof=1)
        vol_of_vol = np.sqrt(var / dt)

        return mr_speed, lt_var, vol_of_vol

    @staticmethod
    def estimate_asset_corr(data: pd.DataFrame, tickers: Sequence[str]) -> np.ndarray:
      n = len(tickers)

      if not isinstance(data.columns, pd.MultiIndex):
        raise ValueError(
          "estimate_asset_corr expected `data` with MultiIndex columns "
          "(PriceType, Ticker), e.g. from yf.download(tickers, ...). "
          f"Got columns: {list(data.columns)[:10]}. "
          "Double-check you didn't accidentally overwrite `data` with "
          "something else (e.g. a VIX series) before calling generate_paths."
        )

      missing = [t for t in tickers if t not in data["Close"].columns]
      if missing:
        raise ValueError(
          f"Tickers {missing} not found in data['Close'].columns "
          f"({list(data['Close'].columns)})."
        )

      close = data["Close"][list(tickers)]
      log_ret = np.log(close / close.shift(1)).dropna()

      if log_ret.shape[0] < 2:
        raise ValueError(
          f"Not enough overlapping return history across {tickers} "
          f"to estimate a correlation matrix (got {log_ret.shape[0]} rows). "
          "Check for NaNs / misaligned dates across tickers."
        )

      corr = log_ret.corr().values
      if corr.shape != (n, n):
        raise ValueError(f"Expected correlation matrix of shape ({n}, {n}), got {corr.shape}.")

      corr = np.nan_to_num(corr, nan=0.0)
      np.fill_diagonal(corr, 1.0)

      eigvals, eigvecs = np.linalg.eigh(corr)
      if np.any(eigvals < 1e-8):
        eigvals_clipped = np.clip(eigvals, 1e-8, None)
        corr = eigvecs @ np.diag(eigvals_clipped) @ eigvecs.T
        d = np.sqrt(np.diag(corr))
        corr = corr / np.outer(d, d)
        np.fill_diagonal(corr, 1.0)

      return corr

    def _calibrate_assets(self, data: pd.DataFrame, vix, tickers: Sequence[str]):
      p = self.params
      n = len(tickers)

      if not isinstance(data.columns, pd.MultiIndex):
        raise ValueError(
          "generate_paths expected `data` with MultiIndex columns "
          "(PriceType, Ticker), e.g. from yf.download(tickers, ...). "
          f"Got columns: {list(data.columns)[:10]}. "
          "Double-check you didn't accidentally overwrite `data` with "
          "something else (e.g. a VIX series) before calling generate_paths."
        )

      k, theta, sigma = self.get_mr_params(vix, p.dt)
      p.mr_speed = np.full(n, k)
      p.lt_var = np.full(n, theta)
      p.vol_of_vol = np.full(n, sigma)

      S0 = np.zeros(n)
      V0 = np.zeros(n)
      for j, ticker in enumerate(tickers):
        asset_data = data.xs(ticker, axis=1, level=1)
        S0[j] = asset_data["Close"].iloc[-1]
        vol0 = self.garman_klass_vol(asset_data)[-1]
        V0[j] = vol0 ** 2

      p.S0 = S0
      p.V0 = V0

      needs_estimate = (
        p.asset_corr is None
        or not isinstance(p.asset_corr, np.ndarray)
        or p.asset_corr.ndim != 2
        or p.asset_corr.shape != (n, n)
      )
      if needs_estimate:
        p.asset_corr = self.estimate_asset_corr(data, tickers)

      self._asset_chol = self._safe_cholesky(p.asset_corr)

    @staticmethod
    def _safe_cholesky(corr: np.ndarray, max_tries: int = 5) -> np.ndarray:
        corr = np.asarray(corr, dtype=float)
        if corr.ndim != 2 or corr.shape[0] != corr.shape[1]:
          raise ValueError(f"asset_corr must be a square matrix, got shape {corr.shape}")

        jitter = 0.0
        n = corr.shape[0]
        for _ in range(max_tries):
          try:
            return np.linalg.cholesky(corr + jitter * np.eye(n))
          except np.linalg.LinAlgError:
            jitter = 1e-10 if jitter == 0.0 else jitter * 10

        eigvals, eigvecs = np.linalg.eigh(corr)
        eigvals_clipped = np.clip(eigvals, 1e-8, None)

        corr_psd = eigvecs @ np.diag(eigvals_clipped) @ eigvecs.T
        d = np.sqrt(np.diag(corr_psd))

        corr_psd = corr_psd / np.outer(d, d)
        np.fill_diagonal(corr_psd, 1.0)

        return np.linalg.cholesky(corr_psd)

    def get_vol(self):
      p = self.params
      shape_full = (p.N, p.S, p.N_assets)
      shape_steps = (p.N, p.S - 1, p.N_assets)

      Z_V = np.random.normal(size=shape_steps)
      V = np.zeros(shape_full)
      V[:, 0, :] = p.V0

      for i in range(p.S - 1):
        V_max = np.maximum(V[:, i, :], 0.0)
        mr_term = p.mr_speed * (p.lt_var - V_max) * p.dt
        vol_term = p.vol_of_vol * np.sqrt(V_max * p.dt) * Z_V[:, i, :]
        V[:, i + 1, :] = V[:, i, :] + mr_term + vol_term

      self.V, self.Z_V = V, Z_V
      return V, Z_V

    def get_jumps(self):
      p = self.params
      shape_steps = (p.N, p.S - 1, p.N_assets)

      poisson_rate = p.jump_intensity * p.dt
      jump_counts = np.random.poisson(lam=poisson_rate, size=shape_steps)

      jump_means = jump_counts * p.mu_j
      jump_stds = np.sqrt(jump_counts) * p.sigma_j

      log_jumps = np.random.normal(loc=jump_means, scale=jump_stds)

      self.log_jumps = log_jumps
      return log_jumps

    def _correlated_asset_shocks(self):
        p = self.params
        Z1 = np.random.normal(size=(p.N, p.S - 1, p.N_assets))
        return Z1 @ self._asset_chol.T

    def generate_paths(self, data: pd.DataFrame, vix, tickers: Sequence[str]) -> np.ndarray:
        p = self.params
        assert len(tickers) == p.N_assets, "tickers must have N_assets entries"

        self._calibrate_assets(data, vix, tickers)

        S = np.zeros((p.N, p.S, p.N_assets))
        S[:, 0, :] = p.S0

        V, Z_V = self.get_vol()

        Z1_corr = self._correlated_asset_shocks()
        rho = p.corr
        Z_S = rho * Z_V + np.sqrt(1 - rho ** 2) * Z1_corr
        self.Z_S = Z_S

        log_jumps = self.get_jumps()

        k_bar = np.exp(p.mu_j + 0.5 * p.sigma_j ** 2) - 1.0
        jump_compensator = p.jump_intensity * k_bar

        for i in range(p.S - 1):
            V_max = np.maximum(V[:, i, :], 0.0)

            drift = (p.rfr - p.div_yield - jump_compensator - 0.5 * V_max) * p.dt
            diffusion = np.sqrt(V_max * p.dt) * Z_S[:, i, :]

            S[:, i + 1, :] = S[:, i, :] * np.exp(drift + diffusion + log_jumps[:, i, :])

        self.S_paths = S
        return S

In [222]:
tickers = ["QLD", "BNDW", "KMLM"]
data = yf.download(tickers, period="1y")
vix = yf.download("^VIX", period="1y")[["Close"]]

params = BatesParams(N=1000, S=252, N_assets=len(tickers))
model = BatesModel(params)
paths = model.generate_paths(data, vix, tickers)

/tmp/ipykernel_3981/4208689457.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, period="1y")
[*********************100%***********************]  3 of 3 completed
/tmp/ipykernel_3981/4208689457.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  vix = yf.download("^VIX", period="1y")[["Close"]]
[*********************100%***********************]  1 of 1 completed
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
